In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [15]:
dataset = [
    ("go left", "MOVE_LEFT"),
    ("move left", "MOVE_LEFT"),
    ("left", "MOVE_LEFT"),
    ("back left", "MOVE_LEFT"),
    ("walk left", "MOVE_LEFT"),
    ("backward", "MOVE_LEFT"),

    ("go right", "MOVE_RIGHT"),
    ("move right", "MOVE_RIGHT"),
    ("right", "MOVE_RIGHT"),
    ("forward", "MOVE_RIGHT"),
    ("move ahead", "MOVE_RIGHT"),

    ("jump", "JUMP"),
    ("hop", "JUMP"),
    ("leap", "JUMP"),
    ("up", "JUMP"),
    ("jump now", "JUMP"),

    ("run", "RUN"),
    ("sprint", "RUN"),
    ("go fast", "RUN"),
    ("speed up", "RUN"),
    ("full speed", "RUN"),

    ("duck", "DUCK"),
    ("crouch", "DUCK"),
    ("go low", "DUCK"),
    ("get down", "DUCK"),
    ("bend down", "DUCK"),

    ("fire", "FIRE"),
    ("shoot", "FIRE"),
    ("throw fireball", "FIRE"),
    ("blast", "FIRE"),
    ("attack", "FIRE"),

    ("stop", "STOP"),
    ("freeze", "STOP"),
    ("hold still", "STOP"),
    ("don't move", "STOP"),
    ("stay", "STOP"),

    ("pause", "PAUSE"),
    ("stop game", "PAUSE"),
    ("pause game", "PAUSE"),
    ("take a break", "PAUSE"),
    ("halt", "PAUSE"),

    ("go down", "ENTER_PIPE"),
    ("enter pipe", "ENTER_PIPE"),
    ("drop down", "ENTER_PIPE"),
    ("go inside", "ENTER_PIPE"),
    ("down pipe", "ENTER_PIPE"),
]

In [16]:
texts = [x[0] for x in dataset]
labels = [x[1] for x in dataset]

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(texts)

In [17]:
model = LogisticRegression()
model.fit(X, labels)

LogisticRegression()

In [18]:
def predict(text):
    x = vectorizer.transform([text])

    probs = model.predict_proba(x)[0]   # probabilities for each class
    best_idx = probs.argmax()           # index of highest probability

    label = model.classes_[best_idx]    # corresponding intent
    confidence = probs[best_idx]        # probability score

    return label, confidence

In [19]:
cvl_memory = {}

In [20]:
import json

def save_memory():
    with open("cvl_memory.json", "w") as f:
        json.dump(cvl_memory, f)

def load_memory():
    global cvl_memory
    try:
        with open("cvl_memory.json", "r") as f:
            cvl_memory = json.load(f)
    except:
        cvl_memory = {}

In [21]:
from difflib import get_close_matches

def semantic_cvl_lookup(text):
    matches = get_close_matches(text, cvl_memory.keys(), n=1, cutoff=0.85)

    if matches:
        return cvl_memory[matches[0]]

    return None

In [22]:
def cvl_predict(text, threshold=0.2):
    text = text.lower().strip()

    if text in cvl_memory:
        return cvl_memory[text], 1.0, "CVL"

    label, confidence = predict(text)

    if confidence >= threshold:
        return label, confidence, "MODEL"

    return None, confidence, "UNKNOWN"

In [23]:
def learn_new_mapping(text, correct_intent):
    cvl_memory[text.lower().strip()] = correct_intent
    save_memory()

In [24]:
def handle_input(text):
    label, conf, source = cvl_predict(text)

    if source != "UNKNOWN":
        print(label, conf, source)
        return label

    # CVL triggers learning
    print(f"Unknown command: {text}")
    user_label = input("What should this mean? ")

    learn_new_mapping(text, user_label)

    return user_label

In [25]:
print(cvl_predict("go left"))
print(cvl_predict("jump now"))
print(cvl_predict("throw fireball"))

(np.str_('MOVE_LEFT'), np.float64(0.40170810179322475), 'MODEL')
(np.str_('JUMP'), np.float64(0.3187216214135115), 'MODEL')
(np.str_('FIRE'), np.float64(0.2451023346132632), 'MODEL')


In [29]:
# testing CVL

test_cases = [
    "cook him",
    "cokk him",
    "cook em",
    "blast him",
    "burn him",
]

for t in test_cases:
    handle_input(t)
    print(t, "→", cvl_predict(t))

cook him → ('ATTACK', 1.0, 'CVL')
cokk him → ('ATTACK', 1.0, 'CVL')
cook em → ('ATTACK', 1.0, 'CVL')
blast him → (np.str_('FIRE'), np.float64(0.2451023346132632), 'MODEL')
burn him → ('FIRE', 1.0, 'CVL')
